# Provider-Only ML Ensembles for LLM Features

Ноутбук отдельно считает ML-ансамбли для каждого LLM-провайдера на уже готовых feature-артефактах.


In [ ]:
from pathlib import Path
import os
import ast
import json
import math
import time
from itertools import combinations

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.base import clone
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import BayesianRidge, ElasticNet, HuberRegressor, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, pairwise_distances
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except Exception as exc:
    XGB_AVAILABLE = False
    print('xgboost недоступен:', exc)

SEED = 2
TARGET_COL = 'duration_hours'
CAP_HOURS = 960
BLEND_TRAIN_FRAC = 0.85
BLEND_FIT_FRAC = 0.5

ROOT = Path.cwd()
OUT_DIR = Path(os.environ.get('PROVIDER_ML_ENSEMBLES_DIR', ROOT / 'artifacts_provider_ml_ensembles')).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)


def require_path(env_name: str, label: str) -> Path:
    raw_value = os.environ.get(env_name, '').strip()
    if not raw_value:
        raise FileNotFoundError(f'Укажи путь к {label} через переменную окружения {env_name}.')
    path = Path(raw_value).expanduser()
    if not path.exists():
        raise FileNotFoundError(f'Файл для {label} не найден: {path}')
    return path


def artifact_dir_from_env(env_name: str, default_path: Path) -> Path:
    return Path(os.environ.get(env_name, default_path)).expanduser()


PROVIDER_SPECS = {
    'gpt': {
        'artifact_dir': artifact_dir_from_env('GPT_LLM_ARTIFACT_DIR', ROOT / 'artifacts'),
        'feature_glob_prefix': 'llm_features',
    },
    'claude': {
        'artifact_dir': artifact_dir_from_env('CLAUDE_LLM_ARTIFACT_DIR', ROOT / 'claude_api' / 'artifacts_claude_api'),
        'feature_glob_prefix': 'llm_features',
    },
    'ollama': {
        'artifact_dir': artifact_dir_from_env('OLLAMA_LLM_ARTIFACT_DIR', ROOT / 'ollama_local' / 'artifacts_ollama_local_pristine'),
        'feature_glob_prefix': 'llm_features',
    },
    'qwen': {
        'artifact_dir': artifact_dir_from_env('QWEN_LLM_ARTIFACT_DIR', ROOT / 'qwen_local' / 'artifacts_qwen_local_pristine'),
        'feature_glob_prefix': 'llm_features',
    },
}
for spec in PROVIDER_SPECS.values():
    spec['summary_path'] = spec['artifact_dir'] / 'llm_cluster_compare_results.csv'

DATA_PATH = require_path('DATA_FINALL_PATH', 'data_finall')


In [ ]:
def read_dataset(path: Path):
    df = pd.read_csv(path)
    if TARGET_COL not in df.columns:
        raise ValueError(f'{TARGET_COL} is absent in {path}')

    # If this is the wider raw export, drop text/date/status columns that earlier ML notebooks excluded.
    drop_cols = ['Ключ', 'Задача', 'Статус', 'Резолюция', 'Создано', 'Дата завершения', 'created_dt']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    return df

df = read_dataset(DATA_PATH)
split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full = df.iloc[split:].copy()
test_typical = test_full[test_full[TARGET_COL] < CAP_HOURS].copy()
train_filt = train_full[train_full[TARGET_COL] < CAP_HOURS].copy()

meta_Xtr0 = train_filt.drop(columns=[TARGET_COL], errors='ignore').reset_index(drop=True)
meta_ytr0 = train_filt[TARGET_COL].reset_index(drop=True)
meta_Xte0 = test_full.drop(columns=[TARGET_COL], errors='ignore').reset_index(drop=True)
meta_yte0 = test_full[TARGET_COL].reset_index(drop=True)
meta_Xte_typ0 = test_typical.drop(columns=[TARGET_COL], errors='ignore').reset_index(drop=True)
meta_yte_typ0 = test_typical[TARGET_COL].reset_index(drop=True)

llm_train_raw = train_filt.reset_index(drop=True).copy()
llm_test_raw = test_full.reset_index(drop=True).copy()
test_typical_mask = (llm_test_raw[TARGET_COL] < CAP_HOURS).reset_index(drop=True)

print('train_full  ', train_full.shape)
print('train_filt  ', train_filt.shape)
print('test_full   ', test_full.shape)
print('test_typical', test_typical.shape)


In [ ]:
def make_clusterer(name, params):
    if name == 'MiniBatchKMeans':
        return MiniBatchKMeans(n_clusters=int(params['n_clusters']), random_state=SEED, n_init=20, batch_size=1024)
    if name == 'GaussianMixture':
        return GaussianMixture(
            n_components=int(params['n_components']),
            covariance_type=params.get('covariance_type', 'diag'),
            random_state=SEED,
        )
    raise ValueError(name)

def probs_from_dist(all_d):
    inv = 1.0 / (all_d + 1e-6)
    return inv / inv.sum(axis=1, keepdims=True)

def fit_clusterer_and_assign(name, params, Xtr, Xte):
    est = make_clusterer(name, params)
    if name == 'MiniBatchKMeans':
        est.fit(Xtr)
        tr_labels = est.predict(Xtr).astype(int)
        te_labels = est.predict(Xte).astype(int)
        tr_d = pairwise_distances(Xtr, est.cluster_centers_)
        te_d = pairwise_distances(Xte, est.cluster_centers_)
        return tr_labels, te_labels, probs_from_dist(tr_d), probs_from_dist(te_d)
    if name == 'GaussianMixture':
        est.fit(Xtr)
        tr_proba = est.predict_proba(Xtr)
        te_proba = est.predict_proba(Xte)
        return np.argmax(tr_proba, axis=1).astype(int), np.argmax(te_proba, axis=1).astype(int), tr_proba, te_proba
    raise ValueError(name)

def build_cluster_meta_features(tr_labels, te_labels, tr_proba=None, te_proba=None):
    tr_feat = pd.DataFrame({'cluster_id': tr_labels})
    te_feat = pd.DataFrame({'cluster_id': te_labels})
    tr_sizes = pd.Series(tr_labels).value_counts().to_dict()
    tr_feat['cluster_size_train'] = pd.Series(tr_labels).map(tr_sizes).fillna(0)
    te_feat['cluster_size_train'] = pd.Series(te_labels).map(tr_sizes).fillna(0)
    tr_feat['is_noise'] = (tr_feat['cluster_id'] == -1).astype(int)
    te_feat['is_noise'] = (te_feat['cluster_id'] == -1).astype(int)

    tr_ohe = pd.get_dummies(tr_feat['cluster_id'], prefix='cluster')
    te_ohe = pd.get_dummies(te_feat['cluster_id'], prefix='cluster').reindex(columns=tr_ohe.columns, fill_value=0)
    tr_feat = pd.concat([tr_feat, tr_ohe], axis=1)
    te_feat = pd.concat([te_feat, te_ohe], axis=1)

    if tr_proba is not None and te_proba is not None:
        for k in range(tr_proba.shape[1]):
            tr_feat[f'cluster_proba_{k}'] = tr_proba[:, k]
            te_feat[f'cluster_proba_{k}'] = te_proba[:, k]
    return tr_feat.reset_index(drop=True), te_feat.reset_index(drop=True)

FIXED_CLUSTER_CONFIGS = {
    'gmm_diag_5': {'clusterer': 'GaussianMixture', 'params': {'n_components': 5, 'covariance_type': 'diag'}},
    'mbkm_2': {'clusterer': 'MiniBatchKMeans', 'params': {'n_clusters': 2}},
}

raw_scaler = StandardScaler()
raw_Xtr_scaled = raw_scaler.fit_transform(meta_Xtr0)
raw_Xte_scaled = raw_scaler.transform(meta_Xte0)
raw_pca = PCA(n_components=min(30, meta_Xtr0.shape[1]), random_state=SEED)
raw_Xtr_embed = raw_pca.fit_transform(raw_Xtr_scaled)
raw_Xte_embed = raw_pca.transform(raw_Xte_scaled)

raw_cluster_features = {}
for tag, cfg in FIXED_CLUSTER_CONFIGS.items():
    tr_labels, te_labels, tr_proba, te_proba = fit_clusterer_and_assign(cfg['clusterer'], cfg['params'], raw_Xtr_embed, raw_Xte_embed)
    raw_cluster_features[tag] = build_cluster_meta_features(tr_labels, te_labels, tr_proba, te_proba)

print('cluster tags:', list(raw_cluster_features))


In [ ]:
def parse_params(raw):
    if isinstance(raw, dict):
        params = raw
    elif pd.isna(raw):
        params = {}
    else:
        text = str(raw).replace('NaN', 'null')
        try:
            params = json.loads(text)
        except Exception:
            params = ast.literal_eval(str(raw))
    return {k: (v.item() if hasattr(v, 'item') else v) for k, v in dict(params).items()}

def apply_params_to_pipeline(pipe, params):
    pipe = clone(pipe)
    names = pipe.get_params(deep=True)
    settable = {}
    for k, v in params.items():
        # Saved sklearn/xgboost params may contain JSON nulls for default values.
        # Re-setting them can break some xgboost versions, so keep estimator defaults.
        if v is None:
            continue
        if f'm__{k}' in names:
            settable[f'm__{k}'] = v
        elif k in names:
            settable[k] = v
    if settable:
        pipe.set_params(**settable)
    return pipe

def model_template(model_name):
    if model_name == 'Ridge':
        return Pipeline([('sc', StandardScaler()), ('m', Ridge(random_state=SEED))])
    if model_name == 'Lasso':
        return Pipeline([('sc', StandardScaler()), ('m', Lasso(random_state=SEED, max_iter=100000))])
    if model_name == 'ElasticNet':
        return Pipeline([('sc', StandardScaler()), ('m', ElasticNet(random_state=SEED, max_iter=100000))])
    if model_name == 'HuberReg':
        return Pipeline([('sc', StandardScaler()), ('m', HuberRegressor(max_iter=10000))])
    if model_name == 'BayRidge':
        return Pipeline([('sc', StandardScaler()), ('m', BayesianRidge())])
    if model_name == 'RF':
        return Pipeline([('m', RandomForestRegressor(random_state=SEED, n_jobs=-1))])
    if model_name == 'ExtraTrees':
        return Pipeline([('m', ExtraTreesRegressor(random_state=SEED, n_jobs=-1))])
    if model_name == 'GBoost':
        return Pipeline([('m', GradientBoostingRegressor(random_state=SEED))])
    if model_name == 'XGB':
        if not XGB_AVAILABLE:
            raise ImportError('xgboost is required for XGB rows')
        return Pipeline([('m', XGBRegressor(objective='reg:absoluteerror', eval_metric='mae', random_state=SEED, tree_method='hist', n_jobs=-1, verbosity=0))])
    if model_name == 'KNN':
        return Pipeline([('sc', StandardScaler()), ('m', KNeighborsRegressor())])
    raise ValueError(model_name)

def normalize_X(df_in):
    out = df_in.copy()
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.fillna(0)
    for c in out.columns:
        if out[c].dtype == 'bool':
            out[c] = out[c].astype(int)
    return out


In [ ]:
def find_feature_file(artifact_dir: Path, experiment: str, split: str) -> Path:
    # split is 'train' or 'test'. GPT uses a short name; other providers include provider/model tags.
    patterns = [
        f'llm_features_{experiment}_{split}.csv',
        f'llm_features_*_{experiment}_{split}.csv',
    ]
    matches = []
    for pattern in patterns:
        matches.extend(sorted(artifact_dir.glob(pattern)))
    matches = sorted(set(matches), key=lambda x: (len(x.name), x.name))
    if not matches:
        raise FileNotFoundError(f'No feature file for {experiment}/{split} in {artifact_dir}; tried {patterns}')
    # Prefer the longest provider-specific filename; avoids generic leftovers when both exist.
    return matches[-1]

def load_llm_features(artifact_dir: Path, experiment: str):
    train_path = find_feature_file(artifact_dir, experiment, 'train')
    test_path = find_feature_file(artifact_dir, experiment, 'test')
    tr = pd.read_csv(train_path).drop(columns=['row_id'], errors='ignore').reset_index(drop=True)
    te = pd.read_csv(test_path).drop(columns=['row_id'], errors='ignore').reset_index(drop=True)
    if len(tr) != len(meta_Xtr0):
        raise ValueError(f'{train_path} rows={len(tr)} expected={len(meta_Xtr0)}')
    if len(te) != len(meta_Xte0):
        raise ValueError(f'{test_path} rows={len(te)} expected={len(meta_Xte0)}')
    return tr, te, train_path, test_path

_FEATURE_CACHE = {}

def build_feature_space(provider: str, experiment: str):
    key = (provider, experiment)
    if key in _FEATURE_CACHE:
        return _FEATURE_CACHE[key]

    artifact_dir = PROVIDER_SPECS[provider]['artifact_dir']
    if experiment == 'llm_only':
        llm_tr, llm_te, tr_path, te_path = load_llm_features(artifact_dir, 'llm_only')
        Xtr = pd.concat([meta_Xtr0.reset_index(drop=True), llm_tr], axis=1)
        Xte_full = pd.concat([meta_Xte0.reset_index(drop=True), llm_te], axis=1)
    elif experiment.startswith('cluster_before_llm__'):
        tag = experiment.replace('cluster_before_llm__', '')
        llm_tr, llm_te, tr_path, te_path = load_llm_features(artifact_dir, experiment)
        cl_tr, cl_te = raw_cluster_features[tag]
        Xtr = pd.concat([meta_Xtr0.reset_index(drop=True), cl_tr, llm_tr], axis=1)
        Xte_full = pd.concat([meta_Xte0.reset_index(drop=True), cl_te, llm_te], axis=1)
    elif experiment.startswith('llm_then_cluster__'):
        tag = experiment.replace('llm_then_cluster__', '')
        llm_tr, llm_te, tr_path, te_path = load_llm_features(artifact_dir, 'llm_only')
        Xtr_llm = pd.concat([meta_Xtr0.reset_index(drop=True), llm_tr], axis=1)
        Xte_llm = pd.concat([meta_Xte0.reset_index(drop=True), llm_te], axis=1)
        cfg = FIXED_CLUSTER_CONFIGS[tag]
        scaler = StandardScaler()
        Xtr_scaled = scaler.fit_transform(normalize_X(Xtr_llm))
        Xte_scaled = scaler.transform(normalize_X(Xte_llm))
        pca = PCA(n_components=min(30, Xtr_llm.shape[1]), random_state=SEED)
        Xtr_embed = pca.fit_transform(Xtr_scaled)
        Xte_embed = pca.transform(Xte_scaled)
        tr_labels, te_labels, tr_proba, te_proba = fit_clusterer_and_assign(cfg['clusterer'], cfg['params'], Xtr_embed, Xte_embed)
        cl_tr, cl_te = build_cluster_meta_features(tr_labels, te_labels, tr_proba, te_proba)
        Xtr = pd.concat([meta_Xtr0.reset_index(drop=True), llm_tr, cl_tr], axis=1)
        Xte_full = pd.concat([meta_Xte0.reset_index(drop=True), llm_te, cl_te], axis=1)
        tr_path = find_feature_file(artifact_dir, 'llm_only', 'train')
        te_path = find_feature_file(artifact_dir, 'llm_only', 'test')
    else:
        raise ValueError(experiment)

    Xtr = normalize_X(Xtr)
    Xte_full = normalize_X(Xte_full)
    Xte_typ = Xte_full.loc[test_typical_mask].reset_index(drop=True)
    payload = (Xtr, Xte_typ, Xte_full, str(tr_path), str(te_path))
    _FEATURE_CACHE[key] = payload
    return payload


In [ ]:
def select_provider_base_rows(provider: str):
    path = PROVIDER_SPECS[provider]['summary_path']
    dfp = pd.read_csv(path)
    required = {'model', 'experiment', 'cv_MAE_internal', 'best_params'}
    missing = required - set(dfp.columns)
    if missing:
        raise ValueError(f'{path} missing columns: {missing}')

    # One best experiment per ML model within provider, matching the logic used for the earlier chart.
    out = (
        dfp.sort_values(['model', 'cv_MAE_internal', 'holdout_typical_MAE', 'holdout_full_MAE'])
           .groupby('model', as_index=False)
           .head(1)
           .sort_values(['cv_MAE_internal', 'holdout_typical_MAE', 'holdout_full_MAE'])
           .reset_index(drop=True)
    )
    out['provider'] = provider
    return out

provider_base_rows = pd.concat([select_provider_base_rows(p) for p in PROVIDER_SPECS], ignore_index=True)
provider_base_rows.to_csv(OUT_DIR / 'provider_selected_base_rows.csv', index=False)
display(provider_base_rows[['provider', 'model', 'experiment', 'cv_MAE_internal', 'holdout_typical_MAE', 'holdout_full_MAE']])


In [ ]:
def metrics(y, pred):
    return {
        'MAE': float(mean_absolute_error(y, pred)),
        'RMSE': float(np.sqrt(mean_squared_error(y, pred))),
        'MdAE': float(median_absolute_error(y, pred)),
    }

def fit_one_row(row):
    provider = row['provider']
    model_name = row['model']
    experiment = row['experiment']
    params = parse_params(row['best_params'])
    Xtr, Xte_typ, Xte_full, train_feature_path, test_feature_path = build_feature_space(provider, experiment)

    n = len(Xtr)
    blend_start = int(n * BLEND_TRAIN_FRAC)
    X_blend_train = Xtr.iloc[:blend_start].reset_index(drop=True)
    y_blend_train = meta_ytr0.iloc[:blend_start].reset_index(drop=True)
    X_blend_val = Xtr.iloc[blend_start:].reset_index(drop=True)
    y_blend_val = meta_ytr0.iloc[blend_start:].reset_index(drop=True)

    pipe = apply_params_to_pipeline(model_template(model_name), params)
    pipe.fit(X_blend_train, y_blend_train)
    pred_blend = pipe.predict(X_blend_val)

    full_pipe = apply_params_to_pipeline(model_template(model_name), params)
    full_pipe.fit(Xtr, meta_ytr0)
    pred_typ = full_pipe.predict(Xte_typ)
    pred_full = full_pipe.predict(Xte_full)

    base_id = f'{provider}::{experiment}::{model_name}'
    return {
        'base_id': base_id,
        'provider': provider,
        'model': model_name,
        'experiment': experiment,
        'train_feature_path': train_feature_path,
        'test_feature_path': test_feature_path,
        'pred_blend_val': pred_blend,
        'pred_typical': pred_typ,
        'pred_full': pred_full,
        'single_blend_MAE': metrics(y_blend_val, pred_blend)['MAE'],
        'single_typical_MAE': metrics(meta_yte_typ0, pred_typ)['MAE'],
        'single_full_MAE': metrics(meta_yte0, pred_full)['MAE'],
    }

base_payloads = []
t0 = time.time()
for _, row in provider_base_rows.iterrows():
    print(f"fit {row['provider']:<7s} {row['model']:<11s} {row['experiment']}")
    base_payloads.append(fit_one_row(row))
print('elapsed min:', round((time.time() - t0) / 60, 2))

base_meta = pd.DataFrame([{k: v for k, v in p.items() if not k.startswith('pred_')} for p in base_payloads])
base_meta.to_csv(OUT_DIR / 'provider_refit_base_metrics.csv', index=False)
display(base_meta.sort_values(['provider', 'single_blend_MAE']))


In [ ]:
def weighted_average(preds, weights):
    mat = np.vstack(preds)
    w = np.asarray(weights, dtype=float)
    w = w / w.sum()
    return np.average(mat, axis=0, weights=w)

def candidate_ensembles_for_provider(payloads, max_subset_size=4):
    rows = []
    payloads = list(payloads)
    yb = meta_ytr0.iloc[int(len(meta_ytr0) * BLEND_TRAIN_FRAC):].reset_index(drop=True)
    blend_cut = int(len(yb) * BLEND_FIT_FRAC)
    y_fit = yb.iloc[:blend_cut].reset_index(drop=True)
    y_sel = yb.iloc[blend_cut:].reset_index(drop=True)

    def split_pred(arr):
        arr = np.asarray(arr)
        return arr[:blend_cut], arr[blend_cut:]

    # Singles are included as a sanity baseline but not used for 'best ensemble' if n>=2.
    for pld in payloads:
        _, pred_sel = split_pred(pld['pred_blend_val'])
        rows.append({
            'provider': pld['provider'],
            'family': 'single',
            'name': 'single',
            'n_models': 1,
            'models': json.dumps([pld['base_id']], ensure_ascii=False),
            'selection_MAE': metrics(y_sel, pred_sel)['MAE'],
            'MAE_typical': metrics(meta_yte_typ0, pld['pred_typical'])['MAE'],
            'MAE_full': metrics(meta_yte0, pld['pred_full'])['MAE'],
            'weights': json.dumps([1.0]),
        })

    max_subset_size = min(max_subset_size, len(payloads))
    for r in range(2, max_subset_size + 1):
        for combo in combinations(payloads, r):
            ids = [pld['base_id'] for pld in combo]
            fit_preds = [split_pred(pld['pred_blend_val'])[0] for pld in combo]
            sel_preds = [split_pred(pld['pred_blend_val'])[1] for pld in combo]

            pred_sel = np.mean(sel_preds, axis=0)
            pred_t = np.mean([pld['pred_typical'] for pld in combo], axis=0)
            pred_f = np.mean([pld['pred_full'] for pld in combo], axis=0)
            rows.append({
                'provider': combo[0]['provider'],
                'family': 'aggregate',
                'name': f'subset{r}_mean',
                'n_models': r,
                'models': json.dumps(ids, ensure_ascii=False),
                'selection_MAE': metrics(y_sel, pred_sel)['MAE'],
                'MAE_typical': metrics(meta_yte_typ0, pred_t)['MAE'],
                'MAE_full': metrics(meta_yte0, pred_f)['MAE'],
                'weights': json.dumps([1.0 / r] * r),
            })

            single_err = np.asarray([metrics(y_fit, pred)['MAE'] for pred in fit_preds])
            inv_w = 1 / np.maximum(single_err, 1e-9)
            pred_sel = weighted_average(sel_preds, inv_w)
            pred_t = weighted_average([pld['pred_typical'] for pld in combo], inv_w)
            pred_f = weighted_average([pld['pred_full'] for pld in combo], inv_w)
            rows.append({
                'provider': combo[0]['provider'],
                'family': 'weighted',
                'name': f'subset{r}_inverse',
                'n_models': r,
                'models': json.dumps(ids, ensure_ascii=False),
                'selection_MAE': metrics(y_sel, pred_sel)['MAE'],
                'MAE_typical': metrics(meta_yte_typ0, pred_t)['MAE'],
                'MAE_full': metrics(meta_yte0, pred_f)['MAE'],
                'weights': json.dumps((inv_w / inv_w.sum()).tolist()),
            })

            if r == 2:
                best = None
                for w0 in np.linspace(0, 1, 101):
                    ws = [float(w0), float(1 - w0)]
                    pred_fit = weighted_average(fit_preds, ws)
                    fit_mae = metrics(y_fit, pred_fit)['MAE']
                    if best is None or fit_mae < best[0]:
                        best = (fit_mae, ws)
                ws = best[1]
                pred_sel = weighted_average(sel_preds, ws)
                pred_t = weighted_average([pld['pred_typical'] for pld in combo], ws)
                pred_f = weighted_average([pld['pred_full'] for pld in combo], ws)
                rows.append({
                    'provider': combo[0]['provider'],
                    'family': 'pair_grid',
                    'name': 'pair_grid',
                    'n_models': r,
                    'models': json.dumps(ids, ensure_ascii=False),
                    'selection_MAE': metrics(y_sel, pred_sel)['MAE'],
                    'MAE_typical': metrics(meta_yte_typ0, pred_t)['MAE'],
                    'MAE_full': metrics(meta_yte0, pred_f)['MAE'],
                    'weights': json.dumps(ws),
                })
    return pd.DataFrame(rows)

leaderboards = []
for provider, group in pd.DataFrame(base_payloads).groupby('provider'):
    payloads = [p for p in base_payloads if p['provider'] == provider]
    lb = candidate_ensembles_for_provider(payloads, max_subset_size=4)
    lb = lb.sort_values(['selection_MAE', 'MAE_typical', 'MAE_full']).reset_index(drop=True)
    lb.to_csv(OUT_DIR / f'{provider}_ml_ensemble_leaderboard.csv', index=False)
    leaderboards.append(lb)

all_leaderboard = pd.concat(leaderboards, ignore_index=True).sort_values(['provider', 'selection_MAE']).reset_index(drop=True)
all_leaderboard.to_csv(OUT_DIR / 'all_provider_ml_ensemble_leaderboard.csv', index=False)

best_provider_ensembles = (
    all_leaderboard[all_leaderboard['n_models'] >= 2]
    .sort_values(['provider', 'selection_MAE', 'MAE_typical', 'MAE_full'])
    .groupby('provider', as_index=False)
    .head(1)
    .reset_index(drop=True)
)
best_provider_ensembles.to_csv(OUT_DIR / 'best_provider_ml_ensembles.csv', index=False)

print('Best provider-only ML ensembles:')
display(best_provider_ensembles)
print('Saved to:', OUT_DIR)


In [ ]:
# Compact row for the chart's ML-ensemble columns.
chart_row = best_provider_ensembles[['provider', 'selection_MAE', 'MAE_typical', 'MAE_full', 'models', 'weights']].copy()
chart_row = chart_row.rename(columns={'selection_MAE': 'ml_ensemble_selection_MAE'})
chart_row.to_csv(OUT_DIR / 'chart_ml_ensemble_provider_values.csv', index=False)
display(chart_row)
